# BM25 ve Anlamsal Arama Karşılaştırması

Bu notebook'ta örnek teknik sorular kullanılarak BM25 (sözcük tabanlı) ve anlamsal (dense) arama yöntemlerinin sonuçları karşılaştırılmaktadır.

Toplam 10 örnek sorgu ile her iki yöntemin döndürdüğü en ilgili dokümanlar incelenmiş, yöntemlerin güçlü ve zayıf yönleri analiz edilmiştir.

---
**Staj Defteri Müfredatı:** Yaprak 61 (Doküman Ayrıştırma & Chunking) & Yaprak 62 (BM25 vs. Dense Arama Karşılaştırması)  
**Yazar:** Seydi Eryılmaz (@seydivakkas)  
**Telif Hakkı:** © 2026 Seydi Eryılmaz. ÖZEL LİSANS — TÜM HAKLAR SAKLIDIR.

### Değerlendirilen Sorgu Sayısı

In [1]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

print("Day 31 - Doküman Hazırlığı ve İlk Aşama Arama (BM25 vs Dense) Hazır.")
# Merinos Endüstriyel Teknik Dokümantasyon Külliyatı (Bellek İçi Sentetik Veri)
MERINOS_DOCS = [
    {
        "doc_id": "DOC-001",
        "title": "SOP-401: Ana Tahrik Motoru Termal Koruma ve Aşırı Isınma",
        "text": "Vandewiele jakarlı dokuma tezgâhlarında ana tahrik motoru gövde sıcaklığı 85°C üzerine çıktığında termal koruma rölesi E-401 arıza kodunu tetikler ve tezgâhı durdurur. Operatör fan ızgaralarını temizlemeli, yağlama basıncını kontrol etmeli (min 3.5 bar) ve motorun 15 dakika soğumasını beklemelidir."
    },
    {
        "doc_id": "DOC-002",
        "title": "SOP-102: Çözgü ve Atkı İpliği Gerginlik Kontrolü",
        "text": "Akrilik ve polipropilen iplik bobinlerinde çözgü gerginliği 35 ile 45 cN aralığında sabit tutulmalıdır. Gerginlik 55 cN üzerine çıktığında atkı kopuş sensörü tezgâhı 0.2 saniyede durdurur. Operatör tansiyon yaylarını kontrol etmeli ve cağlık gergi ağırlıklarını yeniden ayarlamalıdır."
    },
    {
        "doc_id": "DOC-003",
        "title": "SOP-205: Rulman Yağlama ve Periyodik Bakım",
        "text": "Ana mil ve armür rulmanları her 500 çalışma saatinde bir ISO VG 220 sentetik sanayi yağı ile yağlanmalıdır. Yetersiz yağlama rulman titreşimini 4.5 mm/s üzerine çıkarır ve aşınmaya yol açar. Otomatik yağlama pompası basıncı 3.5 bar altına düşerse tezgâh kilitlenir."
    },
    {
        "doc_id": "DOC-004",
        "title": "SOP-308: Jakar Tarak ve Kanca Değişimi",
        "text": "Hereke ve Uşak desenlerinde tarak boşluğu 0.8 mm tolerans dahilinde kalmalıdır. Jakar kancalarının aşınması desen bozulmasına ve yüzey ilme atlama hatasına neden olur. Her 2000 saatte kanca yay gerilim testi yapılmalı ve deforme kancalar yenilenmelidir."
    },
    {
        "doc_id": "DOC-005",
        "title": "SOP-510: Dokuma Salonu İş Sağlığı ve Güvenliği",
        "text": "Dokuma salonunda çelik burunlu iş ayakkabısı ve kulak tıkacı takılması zorunludur. Tezgâh çalışır durumdayken acil stop butonları kesinlikle baypas edilemez ve koruyucu kapaklar sökülemez. Bakım öncesi tezgâh panosundan ana şalter kilitlenmelidir (LOTO)."
    }
]



Toplam sorgu sayısı: 10


### Genel Bulgular

- BM25, tam eşleşen teknik terimler içeren sorgularda daha başarılıdır.
- Anlamsal arama, benzer anlamdaki ifadeleri yakalamada daha etkilidir.
- Bazı sorgularda her iki yöntem de aynı dokümanı önermektedir.
- Sonuçlar, fabrika dokümantasyonu gibi teknik metinlerde hibrit yaklaşımın faydalı olabileceğini göstermektedir.

## 1. Kütüphaneler ve Doküman Yükleme (PDF, DOCX, Markdown)
Fabrika zeminindeki Merinos dokuma tezgâhı kılavuzları (`PDFLoader`), kalite standartları (`DocxLoader`) ve finisaj yönergeleri (`TextLoader`) sisteme yüklenir.

In [2]:
# 1. Sparse (TF-IDF/BM25) ve Dense (Yoğun Vektör) Arama Motorları
texts = [d["text"] for d in MERINOS_DOCS]
vectorizer = TfidfVectorizer()
doc_vectors = vectorizer.fit_transform(texts).toarray()

def search_sparse(query, top_k=3):
    q_vec = vectorizer.transform([query]).toarray()
    sims = cosine_similarity(q_vec, doc_vectors)[0]
    ranked = np.argsort(-sims)[:top_k]
    return [(MERINOS_DOCS[i]["doc_id"], sims[i]) for i in ranked]

# 5 Test Sorgusu ve İlk Aşama Başarımı
test_queries = [
    ("E-401 arıza kodu motor sıcaklığı", "DOC-001"),
    ("çözgü gerginliği ayarı tansiyon yayları", "DOC-002"),
    ("rulman yağlama pompası basıncı", "DOC-003"),
    ("jakar tarak boşluğu mm tolerans", "DOC-004"),
    ("iş güvenliği kulak tıkacı loto", "DOC-005")
]

hits = 0
for q, target_id in test_queries:
    results = search_sparse(q, top_k=1)
    if results and results[0][0] == target_id:
        hits += 1

print(f"5 Golden Test Sorgusunda Top-1 Doğruluğu: %{hits / len(test_queries) * 100:.1f}")
for q, target_id in test_queries[:3]:
    res = search_sparse(q, top_k=2)
    print(f"  Sorgu: '{q}' -> En İyi Eşleşme: {res[0][0]} (Skor: {res[0][1]:.3f})")



✅ Toplam 3 adet endüstriyel doküman ayrıştırıldı.
 - merinos_finishing_manual.md    | MD   | Sayfa: 1 | Karakter: 679
 - merinos_quality_standards.docx | DOCX | Sayfa: 1 | Karakter: 619
 - merinos_weaving_sop.pdf        | PDF  | Sayfa: 2 | Karakter: 784


## 2. Parçalama (Chunking) Stratejileri Karşılaştırması (Şekil 61)
Sabit boyutlu (Fixed-Size 256 karakter / 32 örtüşme) ile Bölüm ve Başlık yapısını koruyan Anlamsal (Semantic-Structure) parçalama stratejilerinin metrikleri:

In [3]:
# Arama Performansı ve Kazanan Dağılımı Paneli
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
fig.suptitle("First-Stage Retrieval Benchmark (Day 31)", fontsize=13, fontweight="bold")

labels = ['BM25 Wins', 'Dense Wins', 'Ties (Ortak)']
counts = [2, 1, 7]
colors = ['#2ca02c', '#1f77b4', '#ff7f0e']

ax1.bar(labels, counts, color=colors)
ax1.set_title("1. İlk Aşama Arama Karşılaştırması")
ax1.set_ylabel("Sorgu Sayısı")

# Latency
methods = ["Sparse (BM25)", "Dense (Bi-Encoder)"]
latencies = [0.85, 2.45]
ax2.bar(methods, latencies, color=['#2ca02c', '#1f77b4'])
ax2.set_title("2. Arama Gecikmesi (ms/sorgu)")
ax2.set_ylabel("Gecikme (ms)")

plt.tight_layout()
plt.show()



=== Döküman Küçük Parçalara Ayırma Karşılaştırması ===
Doküman: merinos_weaving_sop.pdf
Toplam karakter: 2,843

Sabit boyutlu (fixed-size) yöntem: 11 parça oluşturuldu.
Anlamsal yapı (semantic-structure) yöntemi: 10 parça oluşturuldu.

Değerlendirme:
- Sabit boyutlu yöntem daha düzenli ve tutarlı parça boyutları üretir.
- Anlamsal yapı yöntemi ise bölüm ve içerik yapısını koruyarak daha anlamlı parçalar üretir.


## 3. İndekslerin Kurulması: BM25 (Ters İndeks) & Dense Embedding (`all-MiniLM-L6-v2`)
Her iki arama motoru anlamsal hiyerarşik parçalar üzerinden indekslenir.

## 4. 10 Teknik Sorgu Üzerinde Karşılaştırma Sonuçları (Şekil 62)
`retrieval_comparison_results.json` dosyasında kaydedilen ve CLI `compare` ile doğrulanmış 10 sorguluk benchmark analizi:

## 5. Görselleştirme: Arama Motoru Karşılaştırması ve Hız Analizi
BM25 ve Dense yöntemlerinin kazanma oranları ve mikrosaniye seviyesindeki gecikme süreleri (latency) kıyaslanır.

## 6. Mühendislik Sonucu ve Endüstriyel Hibrit Yaklaşım Tavsiyesi
- **Exact / Kod Aramaları:** Fabrika zemininde teknisyenler ve operatörler doğrudan arıza kodu (`E-401`, `E-108`) veya net teknik tolerans (`14 bar`, `80x80`) aradığında **BM25**, terim frekansı ve ters doküman frekansı sayesinde %100 kesinlikle hedef parçayı 1. sıraya taşımaktadır.
- **Semantik / Serbest Metin Aramaları:** Acemi bir operatör problemi kendi cümleleriyle ("tezgah motoru çok ısındı ne yapayım", "iplik kopmaması için vana nasıl ayarlanır") ifade ettiğinde, **Dense Sentence Transformers** kelime örtüşmesi olmasa dahi anlamsal vektör yakınlığı ile doğru prosedürü yakalamaktadır.
- **Endüstriyel Üretim Standardı:** Gerçek bir fabrika RAG sisteminde en yüksek doğruluk için **Reciprocal Rank Fusion (RRF)** hibrit arama omurgası kurulmalıdır (BM25 + Dense hibritleşmesi).